# Generalized Multi-Position Pipeline

In [17]:
import numpy as np
import pandas as pd
from pathlib import Path

from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GroupKFold, GridSearchCV
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

In [18]:
DATA_PATH = Path('../artifacts/roster_roi_combined.csv')
OUTPUT_PATH = Path('../artifacts/roster_roi_scored.csv')

league_cap_by_year = {
    2021: 182.5,
    2022: 208.2,
    2023: 224.8,
    2024: 255.4,
    2025: 279.2
}

RANDOM_STATE = 42
POSITIONS = ['QB', 'RB', 'WR', 'TE']
MIN_SNAPS = 100

features = ['total_epa', 'snaps', 'epa_per_snap', 'age', 'years_exp']

In [19]:
df = pd.read_csv(DATA_PATH)
df['target_pct'] = df['cap_pct_of_team'].astype(float) * 100.0
df['target_pct'] = df['target_pct'].replace([np.inf, -np.inf], np.nan).fillna(0.0)
df['y_log'] = np.log1p(df['target_pct'])

median_cap = np.nanmedian(list(league_cap_by_year.values()))
df['season_cap_m'] = df['season'].map(league_cap_by_year).fillna(median_cap)

In [20]:
scored_dataframes = []
print('NFL Expected APY Modeling (5-Year Veteran Markets)\n')
for pos in POSITIONS:
    pos_df = df[(df['position'] == pos) & (df['snaps'].fillna(0) >= MIN_SNAPS)].copy()
    
    train_df = pos_df[pos_df['is_rookie_deal'] == False].copy()
    
    X_train = train_df[features].fillna(0).astype(float)
    y_train_log = train_df['y_log'].values
    groups = train_df['gsis_id'].values
    
    max_train_epa = X_train['total_epa'].max()
    max_train_snaps = X_train['snaps'].max()
    max_market_apy = train_df['yearly_cap_hit'].max() 

    gkf = GroupKFold(n_splits=5)
    pipe = Pipeline([
        ('scaler', StandardScaler()),
        ('ridge', Ridge(random_state=RANDOM_STATE))
    ])
    
    param_grid = {'ridge__alpha': np.logspace(-3, 3, 50)}
    gs = GridSearchCV(pipe, param_grid=param_grid, cv=gkf, n_jobs=-1, scoring='neg_root_mean_squared_error')
    gs.fit(X_train, y_train_log, groups=groups)
    
    best_model = gs.best_estimator_
    
    train_preds_log = best_model.predict(X_train)
    train_preds_pct = np.expm1(train_preds_log)
    train_season_caps = train_df['season'].map(league_cap_by_year).ffill()
    train_preds_dollars = (train_preds_pct / 100.0) * train_season_caps
    r2 = r2_score(train_df['yearly_cap_hit'], train_preds_dollars)
    mae = mean_absolute_error(train_df['yearly_cap_hit'], train_preds_dollars)
    
    print(f"[{pos}] R2: {r2:.3f} | MAE: ${mae:.2f}M | Optimal Alpha: {gs.best_params_['ridge__alpha']:.2f}")

    X_all = pos_df[features].fillna(0).astype(float)
    all_preds_log = best_model.predict(X_all)
    all_preds_pct = np.expm1(all_preds_log)
    
    raw_expected_dollars = (all_preds_pct / 100.0) * pos_df['season_cap_m']
    
    market_ceiling = max_market_apy * 1.10
    pos_df['expected_apy'] = np.clip(raw_expected_dollars, a_min=0, a_max=market_ceiling)
    pos_df['surplus_value'] = pos_df['expected_apy'] - pos_df['yearly_cap_hit']
    pos_df['is_extrapolated'] = (pos_df['total_epa'] > max_train_epa) | (pos_df['snaps'] > max_train_snaps)
    
    scored_dataframes.append(pos_df)

NFL Expected APY Modeling (5-Year Veteran Markets)

[QB] R2: 0.369 | MAE: $11.04M | Optimal Alpha: 8.29
[RB] R2: 0.428 | MAE: $2.20M | Optimal Alpha: 14.56
[WR] R2: 0.404 | MAE: $4.74M | Optimal Alpha: 3.56
[TE] R2: 0.505 | MAE: $2.38M | Optimal Alpha: 8.29


In [21]:
final_scored_df = pd.concat(scored_dataframes, ignore_index=True)
final_scored_df.to_csv(OUTPUT_PATH, index=False)
print(f"\nSuccessfully scored {len(final_scored_df)} player-seasons. Saved to {OUTPUT_PATH}")


Successfully scored 2168 player-seasons. Saved to ../artifacts/roster_roi_scored.csv


In [22]:
print("\nTop 2 Arbitrage Assets by Position (2025)")
check_df = final_scored_df[final_scored_df['season'] == 2025]

for pos in POSITIONS:
    top_pos = check_df[check_df['position'] == pos].sort_values('surplus_value', ascending=False).head(2)
    print(f"\n-- {pos} --")
    for _, r in top_pos.iterrows():
        cohort = 'Rookie' if r['is_rookie_deal'] else 'Veteran'
        extrap = '[EXTRAPOLATED]' if r['is_extrapolated'] else ''
        print(f"{r['player_name']} ({cohort}) | Actual: ${r['yearly_cap_hit']:.2f}M | Expected: ${r['expected_apy']:.2f}M | Surplus: +${r['surplus_value']:.2f}M {extrap}")


Top 2 Arbitrage Assets by Position (2025)

-- QB --
Drake Maye (Rookie) | Actual: $9.16M | Expected: $66.00M | Surplus: +$56.84M 
Caleb Williams (Rookie) | Actual: $9.87M | Expected: $57.39M | Surplus: +$47.52M 

-- RB --
Chase Brown (Rookie) | Actual: $1.03M | Expected: $9.05M | Surplus: +$8.02M 
Javonte Williams (Veteran) | Actual: $3.00M | Expected: $10.53M | Surplus: +$7.53M 

-- WR --
Puka Nacua (Rookie) | Actual: $1.02M | Expected: $29.10M | Surplus: +$28.08M 
George Pickens (Rookie) | Actual: $1.69M | Expected: $24.45M | Surplus: +$22.76M 

-- TE --
A.J. Barner (Rookie) | Actual: $1.19M | Expected: $16.48M | Surplus: +$15.29M 
Cade Otton (Rookie) | Actual: $1.13M | Expected: $13.65M | Surplus: +$12.52M 
